# ODER Black-Hole Verification Artifact

This notebook verifies the black-hole ODER retrieval analysis for the current observer-indexed proper-time saturation model.

It is built for readers who want to rerun, inspect, challenge, or extend the analysis. The notebook keeps the scientific object fixed: observer-indexed entropy retrieval under a bounded proper-time law. Local setup can change. The analysis preset should not change unless a new analysis version is declared.

## Version Lock

Artifact: `ODER_BH_verification_artifact_v2`  
Version: `2.0`  
Analysis preset: `ODER_BH_RETRIEVAL_V2`  
Run mode: `publication`  
Bootstrap traces: `200`  
Bond dimensions: `D=4,8`  
Seed: `20260614`  
Generated: `2026-06-14`

Paper section supported: black-hole retrieval law, observer-class signatures, bootstrap bands, D=4/D=8 finite-resolution robustness, and adversarial falsification controls.

This notebook verifies synthetic/proxy signatures only. It does not ingest astrophysical data, does not claim explicit tensor-network contractions or microscopic black-hole dynamics, and does not introduce new physical models.

## Validation Scope

This notebook validates numerical consistency, 90% Retrieval Horizon ordering, inverse retrieval recovery, finite-resolution robustness, and adversarial-null discrimination.

It does not constitute empirical confirmation.

## Reader Map

The notebook has six layers.

1. Environment, setup, and preset. Paths, outputs, frozen scientific parameters, and observer definitions.
2. Retrieval engine. The ODER law, inverse map, analytic check, and observer summaries.
3. Robustness checks. Bootstrap bands and D=4/D=8 finite-resolution proxy checks.
4. Adversarial nulls. Generic saturation, Shared-Observer Envelope Null, Observer-Label Permutation Null, Proper-Time Jitter Null, and Non-Gap Dynamics Null.
5. Figures and tables. All artifacts are written after the analysis objects exist.
6. Verification report. A claim-to-artifact map with pass/warn flags and boundary notes.

## 0. Environment Bootstrap

This notebook uses only `numpy` and `matplotlib`. If a package is missing, install it into the active kernel before rerunning. The verification logic itself does not depend on external data.

In [1]:
from pathlib import Path
import json
import math
import platform
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_ARTIFACT = "ODER_BH_verification_artifact_v2"
NOTEBOOK_VERSION = "2.0"
RUN_MODE = "publication"  # "quick" or "publication"
ANALYSIS_PRESET = "ODER_BH_RETRIEVAL_V2"
RANDOM_SEED = 20260614

cwd = Path.cwd()
REPO_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
OUTPUT_DIR = REPO_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for p in (OUTPUT_DIR, FIGURE_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

def artifact_path(path):
    path = Path(path).resolve()
    try:
        return path.relative_to(REPO_ROOT.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


rng = np.random.default_rng(RANDOM_SEED)

MANIFEST = {
    "notebook_artifact": NOTEBOOK_ARTIFACT,
    "notebook_version": NOTEBOOK_VERSION,
    "analysis_preset": ANALYSIS_PRESET,
    "run_mode": RUN_MODE,
    "random_seed": RANDOM_SEED,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "artifacts": {},
    "pass_flags": {},
    "thresholds": {},
    "nulls_executed": [],
}

print(f"Notebook artifact: {NOTEBOOK_ARTIFACT}")
print(f"Version: {NOTEBOOK_VERSION}")
print(f"Analysis preset: {ANALYSIS_PRESET}")
print(f"Output directory: {artifact_path(OUTPUT_DIR)}")

Notebook artifact: ODER_BH_verification_artifact_v2
Version: 2.0
Analysis preset: ODER_BH_RETRIEVAL_V2
Output directory: outputs


## 1. Setup and Frozen Retrieval Preset

This section is the analysis contract. It defines the observer classes, time grid, retrieval threshold, bootstrap count, and finite-resolution proxy settings used below.

Changing these values creates a new analysis version.

In [2]:
S_MAX = 1.0
TAU_RH_THRESHOLD = 0.90
TIME_MAX = 50.0
DT = 0.1
TIME = np.arange(0.0, TIME_MAX + DT, DT)

N_BOOTSTRAP = 50 if RUN_MODE == "quick" else 200
N_PERMUTATIONS = 200 if RUN_MODE == "quick" else 2000
N_JITTER = 50 if RUN_MODE == "quick" else 300
BOOTSTRAP_GAMMA_NOISE = 0.015
JITTER_SIGMA = 0.35
BOND_DIMENSIONS = [4, 8]

ORDERING_PRESERVATION_THRESHOLD = 0.95
INVERSE_RECOVERY_ERROR_THRESHOLD = 0.005
MATCHED_NULL_ALLOWED_FULL_SIGNATURES = 0
NON_GAP_MAX_STABLE_INVERSES = 1
LABEL_PERMUTATION_BASELINE_TOLERANCE = 0.05
INVERSE_FINITE_FRACTION_THRESHOLD = 0.55
INVERSE_NEGATIVE_FRACTION_MAX = 0.02
INVERSE_GAMMA_CV_MAX = 1.25
INVERSE_ROUGHNESS_MAX = 0.035

PREDICTED_ORDER = ["free_falling", "accelerating", "stationary"]

def gamma_stationary(t):
    return np.full_like(t, 0.085, dtype=float)

def gamma_free_falling(t):
    return 0.04 + 0.38 / (1.0 + np.exp(-(t - 8.0) / 0.9))

def gamma_accelerating(t):
    return 0.08 + 0.004 * t

OBSERVER_PRESET = {
    "stationary": dict(
        label="Stationary",
        tau_char=5.0,
        gamma_fn=gamma_stationary,
        description="fixed-radius observer with slower modular access",
    ),
    "free_falling": dict(
        label="Free falling",
        tau_char=2.0,
        gamma_fn=gamma_free_falling,
        description="geodesic observer with post-crossing access increase",
    ),
    "accelerating": dict(
        label="Accelerating",
        tau_char=3.0,
        gamma_fn=gamma_accelerating,
        description="accelerating observer with Unruh-assisted retrieval",
    ),
}

print(json.dumps({
    "S_MAX": S_MAX,
    "TAU_RH_THRESHOLD": TAU_RH_THRESHOLD,
    "TIME_MAX": TIME_MAX,
    "DT": DT,
    "N_BOOTSTRAP": N_BOOTSTRAP,
    "N_PERMUTATIONS": N_PERMUTATIONS,
    "N_JITTER": N_JITTER,
    "BOND_DIMENSIONS": BOND_DIMENSIONS,
    "thresholds": {
        "ordering_preservation": ORDERING_PRESERVATION_THRESHOLD,
        "inverse_recovery_error": INVERSE_RECOVERY_ERROR_THRESHOLD,
        "matched_null_allowed_full_signatures": MATCHED_NULL_ALLOWED_FULL_SIGNATURES,
        "non_gap_max_stable_inverses": NON_GAP_MAX_STABLE_INVERSES,
        "label_permutation_baseline_tolerance": LABEL_PERMUTATION_BASELINE_TOLERANCE,
        "inverse_finite_fraction_min": INVERSE_FINITE_FRACTION_THRESHOLD,
        "inverse_negative_fraction_max": INVERSE_NEGATIVE_FRACTION_MAX,
        "inverse_gamma_cv_max": INVERSE_GAMMA_CV_MAX,
        "inverse_roughness_max": INVERSE_ROUGHNESS_MAX,
    },
    "predicted_order": PREDICTED_ORDER,
    "observers": {k: {"tau_char": v["tau_char"], "description": v["description"]} for k, v in OBSERVER_PRESET.items()},
}, indent=2))

{
  "S_MAX": 1.0,
  "TAU_RH_THRESHOLD": 0.9,
  "TIME_MAX": 50.0,
  "DT": 0.1,
  "N_BOOTSTRAP": 200,
  "N_PERMUTATIONS": 2000,
  "N_JITTER": 300,
  "BOND_DIMENSIONS": [
    4,
    8
  ],
  "thresholds": {
    "ordering_preservation": 0.95,
    "inverse_recovery_error": 0.005,
    "matched_null_allowed_full_signatures": 0,
    "non_gap_max_stable_inverses": 1,
    "label_permutation_baseline_tolerance": 0.05,
    "inverse_finite_fraction_min": 0.55,
    "inverse_negative_fraction_max": 0.02,
    "inverse_gamma_cv_max": 1.25,
    "inverse_roughness_max": 0.035
  },
  "predicted_order": [
    "free_falling",
    "accelerating",
    "stationary"
  ],
  "observers": {
    "stationary": {
      "tau_char": 5.0,
      "description": "fixed-radius observer with slower modular access"
    },
    "free_falling": {
      "tau_char": 2.0,
      "description": "geodesic observer with post-crossing access increase"
    },
    "accelerating": {
      "tau_char": 3.0,
      "description": "accelerating

## 1.1 Pass/Fail Thresholds

All pass criteria are declared before the analysis runs.

| Metric | PASS threshold |
|---|---:|
| Bootstrap traces | 200 in publication mode |
| 90% Retrieval Horizon ordering | free-falling < accelerating < stationary |
| Inverse recovery median absolute error | < 0.005 |
| Inverse finite fraction | > 0.55 |
| Inverse negative fraction | < 0.02 |
| Inverse gamma coefficient of variation | < 1.25 |
| Inverse roughness | < 0.035 |
| D=4/D=8 robustness | ordering, boundedness, and monotonicity preserved for both D values |
| Matched Saturating Envelope Null | 0 families pass the full ODER signature set |
| Shared-Observer Envelope Null | shared envelope fails observer-class separation |
| Observer-Label Permutation Null | measured strict-ordering chance rate within 0.05 of 1/6 baseline |
| Proper-Time Jitter Null | ordering preserved in at least 95% of jitter samples |
| Non-Gap Dynamics Null | at most 1 observer yields stable inverse recovery |

In [3]:
MANIFEST["thresholds"] = {
    "bootstrap_traces_publication": 200,
    "retrieval_horizon_order": PREDICTED_ORDER,
    "inverse_recovery_median_abs_error_lt": INVERSE_RECOVERY_ERROR_THRESHOLD,
    "ordering_preservation_ge": ORDERING_PRESERVATION_THRESHOLD,
    "matched_null_allowed_full_signatures": MATCHED_NULL_ALLOWED_FULL_SIGNATURES,
    "non_gap_max_stable_inverses": NON_GAP_MAX_STABLE_INVERSES,
    "label_permutation_baseline_tolerance": LABEL_PERMUTATION_BASELINE_TOLERANCE,
    "inverse_finite_fraction_min": INVERSE_FINITE_FRACTION_THRESHOLD,
    "inverse_negative_fraction_max": INVERSE_NEGATIVE_FRACTION_MAX,
    "inverse_gamma_cv_max": INVERSE_GAMMA_CV_MAX,
    "inverse_roughness_max": INVERSE_ROUGHNESS_MAX,
    "bond_dimensions": BOND_DIMENSIONS,
}
print(json.dumps(MANIFEST["thresholds"], indent=2))

{
  "bootstrap_traces_publication": 200,
  "retrieval_horizon_order": [
    "free_falling",
    "accelerating",
    "stationary"
  ],
  "inverse_recovery_median_abs_error_lt": 0.005,
  "ordering_preservation_ge": 0.95,
  "matched_null_allowed_full_signatures": 0,
  "non_gap_max_stable_inverses": 1,
  "label_permutation_baseline_tolerance": 0.05,
  "inverse_finite_fraction_min": 0.55,
  "inverse_negative_fraction_max": 0.02,
  "inverse_gamma_cv_max": 1.25,
  "inverse_roughness_max": 0.035,
  "bond_dimensions": [
    4,
    8
  ]
}


## 2. Signature Registry

The registry is the contract between the retrieval law and the verification metrics.

Each signature is named before the analysis runs. Core signatures should pass for ODER traces. Adversarial controls are expected to fail one or more signatures, unless the control preserves the relevant structure by construction.

In [4]:
SIGNATURES = {
    "bounded": "0 <= S_ret(tau) <= S_max for all tau.",
    "monotone": "S_ret(tau) is non-decreasing.",
    "retrieval_horizon_order": "90% Retrieval Horizon ordering is free_falling < accelerating < stationary.",
    "inverse_gamma_stable": "ODER inverse map returns finite, mostly non-negative, not excessively rough gamma(tau).",
    "bootstrap_bounded": "Bootstrap traces remain bounded under spectral noise.",
    "d4_d8_robust": "D=4 and D=8 finite-resolution proxy traces preserve ordering and boundedness.",
    "generic_saturation_rejected": "Matched generic saturation families do not reproduce the full ODER signature set.",
    "shared_observer_envelope_rejected": "A shared underlying envelope with amplitude/noise variation does not reproduce observer-class separation.",
    "label_permutation_quantified": "Random label assignment establishes the chance baseline for observer ordering.",
    "time_jitter_robust": "Ordering survives reasonable proper-time jitter.",
    "non_gap_dynamics_rejected": "Non-gap dynamics yield degraded ODER inverse-map diagnostics.",
}

print(json.dumps(SIGNATURES, indent=2))

{
  "bounded": "0 <= S_ret(tau) <= S_max for all tau.",
  "monotone": "S_ret(tau) is non-decreasing.",
  "retrieval_horizon_order": "90% Retrieval Horizon ordering is free_falling < accelerating < stationary.",
  "inverse_gamma_stable": "ODER inverse map returns finite, mostly non-negative, not excessively rough gamma(tau).",
  "bootstrap_bounded": "Bootstrap traces remain bounded under spectral noise.",
  "d4_d8_robust": "D=4 and D=8 finite-resolution proxy traces preserve ordering and boundedness.",
  "generic_saturation_rejected": "Matched generic saturation families do not reproduce the full ODER signature set.",
  "shared_observer_envelope_rejected": "A shared underlying envelope with amplitude/noise variation does not reproduce observer-class separation.",
  "label_permutation_quantified": "Random label assignment establishes the chance baseline for observer ordering.",
  "time_jitter_robust": "Ordering survives reasonable proper-time jitter.",
  "non_gap_dynamics_rejected": "Non

## 3. Preset Integrity Check

Every verification run starts by checking the preset. These checks validate the notebook configuration, not a result.

In [5]:
def check_preset():
    rows = []
    rows.append(("observer_classes_present", set(OBSERVER_PRESET) == {"stationary", "free_falling", "accelerating"}))
    rows.append(("time_grid_increasing", bool(np.all(np.diff(TIME) > 0))))
    rows.append(("time_grid_finite", bool(np.isfinite(TIME).all())))
    rows.append(("bootstrap_count_positive", N_BOOTSTRAP > 0))
    rows.append(("permutation_count_positive", N_PERMUTATIONS > 0))
    rows.append(("bond_dimensions_declared", BOND_DIMENSIONS == [4, 8]))
    for name, cfg in OBSERVER_PRESET.items():
        gamma = cfg["gamma_fn"](TIME)
        rows.append((f"{name}_tau_char_positive", cfg["tau_char"] > 0))
        rows.append((f"{name}_gamma_finite", bool(np.isfinite(gamma).all())))
        rows.append((f"{name}_gamma_nonnegative", bool((gamma >= 0).all())))
    MANIFEST["pass_flags"]["preset_integrity"] = all(ok for _, ok in rows)
    for key, ok in rows:
        print(f"{'PASS' if ok else 'WARN'}  {key}")
    return rows

preset_integrity = check_preset()

PASS  observer_classes_present
PASS  time_grid_increasing
PASS  time_grid_finite
PASS  bootstrap_count_positive
PASS  permutation_count_positive
PASS  bond_dimensions_declared
PASS  stationary_tau_char_positive
PASS  stationary_gamma_finite
PASS  stationary_gamma_nonnegative
PASS  free_falling_tau_char_positive
PASS  free_falling_gamma_finite
PASS  free_falling_gamma_nonnegative
PASS  accelerating_tau_char_positive
PASS  accelerating_gamma_finite
PASS  accelerating_gamma_nonnegative


## 4. Retrieval Engine

The engine implements the current ODER law:

\[
\frac{dS_{\rm ret}}{d\tau}=\gamma(\tau)\,[S_{\max}-S_{\rm ret}(\tau)]\tanh(\tau/\tau_{\rm char}).
\]

The inverse map is applied only after traces are generated.

In [6]:
def integrate_retrieval(time, gamma_values, tau_char, s_max=S_MAX, s0=0.0):
    s = np.zeros_like(time, dtype=float)
    s[0] = s0
    for i in range(1, len(time)):
        dt = time[i] - time[i - 1]
        d_s = gamma_values[i - 1] * (s_max - s[i - 1]) * math.tanh(time[i - 1] / tau_char)
        s[i] = min(s_max, max(0.0, s[i - 1] + d_s * dt))
    return s

def exact_constant_gamma_solution(time, gamma, tau_char, s_max=S_MAX, s0=0.0):
    integral = tau_char * np.log(np.cosh(time / tau_char))
    return s_max - (s_max - s0) * np.exp(-gamma * integral)

def invert_gamma(time, s_ret, tau_char, s_max=S_MAX):
    d_s = np.gradient(s_ret, time)
    denom = (s_max - s_ret) * np.tanh(time / tau_char)
    gamma = np.divide(d_s, denom, out=np.full_like(s_ret, np.nan), where=np.abs(denom) > 1e-8)
    gamma[(time < max(time[1], tau_char / 10.0)) | (s_ret > 0.98 * s_max)] = np.nan
    return gamma

def tau_rh(time, s_ret, threshold=TAU_RH_THRESHOLD, s_max=S_MAX):
    idx = np.where(s_ret >= threshold * s_max)[0]
    return None if len(idx) == 0 else float(time[idx[0]])

def trace_metrics(time, s_ret, tau_char):
    gamma_inv = invert_gamma(time, s_ret, tau_char)
    finite = np.isfinite(gamma_inv)
    if finite.sum() >= 3:
        gi = gamma_inv[finite]
        roughness = float(np.nanmedian(np.abs(np.diff(gi))))
        neg_fraction = float(np.mean(gi < -1e-8))
        finite_fraction = float(np.mean(finite))
        gamma_median = float(np.nanmedian(gi))
        gamma_cv = float(np.nanstd(gi) / (abs(gamma_median) + 1e-9))
    else:
        roughness = float("inf")
        neg_fraction = 1.0
        finite_fraction = float(np.mean(finite))
        gamma_median = float("nan")
        gamma_cv = float("inf")
    return {
        "bounded": bool((s_ret >= -1e-12).all() and (s_ret <= S_MAX + 1e-12).all()),
        "monotone": bool((np.diff(s_ret) >= -1e-12).all()),
        "tau_rh": tau_rh(time, s_ret),
        "final_entropy": float(s_ret[-1]),
        "inverse_finite_fraction": finite_fraction,
        "inverse_negative_fraction": neg_fraction,
        "inverse_roughness": roughness,
        "inverse_gamma_median": gamma_median,
        "inverse_gamma_cv": gamma_cv,
    }

def inverse_stable(metrics):
    return (
        metrics["inverse_finite_fraction"] > INVERSE_FINITE_FRACTION_THRESHOLD
        and metrics["inverse_negative_fraction"] < INVERSE_NEGATIVE_FRACTION_MAX
        and metrics["inverse_gamma_cv"] < INVERSE_GAMMA_CV_MAX
        and metrics["inverse_roughness"] < INVERSE_ROUGHNESS_MAX
    )

def save_csv(path, header, rows):
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        f.write(",".join(header) + "\n")
        for row in rows:
            f.write(",".join(str(x) for x in row) + "\n")
    return path

def order_pass_from_metrics(metrics_by_observer):
    vals = [metrics_by_observer[name]["tau_rh"] for name in PREDICTED_ORDER]
    return all(v is not None for v in vals) and all(a < b for a, b in zip(vals, vals[1:]))

## 5. Core ODER Analysis

In [7]:
core = {}
for name, cfg in OBSERVER_PRESET.items():
    gamma = cfg["gamma_fn"](TIME)
    s_ret = integrate_retrieval(TIME, gamma, cfg["tau_char"])
    metrics = trace_metrics(TIME, s_ret, cfg["tau_char"])
    gamma_inv = invert_gamma(TIME, s_ret, cfg["tau_char"])
    finite = np.isfinite(gamma_inv)
    metrics["inverse_gamma_median_abs_error"] = float(np.nanmedian(np.abs(gamma_inv[finite] - gamma[finite])))
    metrics["inverse_recovery_pass"] = metrics["inverse_gamma_median_abs_error"] < INVERSE_RECOVERY_ERROR_THRESHOLD
    core[name] = {"gamma": gamma, "s_ret": s_ret, "gamma_inv": gamma_inv, "metrics": metrics}

core_rows = []
for name in PREDICTED_ORDER:
    m = core[name]["metrics"]
    core_rows.append([
        name,
        OBSERVER_PRESET[name]["tau_char"],
        m["tau_rh"],
        m["final_entropy"],
        m["bounded"],
        m["monotone"],
        m["inverse_recovery_pass"],
        m["inverse_gamma_median_abs_error"],
    ])
    print(core_rows[-1])

core_summary_csv = save_csv(
    TABLE_DIR / "core_observer_summary.csv",
    ["observer", "tau_char", "tau_rh", "final_entropy", "bounded", "monotone", "inverse_recovery_pass", "inverse_gamma_median_abs_error"],
    core_rows,
)
MANIFEST["artifacts"]["core_observer_summary"] = artifact_path(core_summary_csv)

MANIFEST["pass_flags"]["core_bounded"] = all(core[name]["metrics"]["bounded"] for name in core)
MANIFEST["pass_flags"]["core_monotone"] = all(core[name]["metrics"]["monotone"] for name in core)
MANIFEST["pass_flags"]["core_order"] = order_pass_from_metrics({k: v["metrics"] for k, v in core.items()})
MANIFEST["pass_flags"]["core_inverse_recovery"] = all(core[name]["metrics"]["inverse_recovery_pass"] for name in core)

['free_falling', 2.0, 12.9, 0.9999999882160501, True, True, True, 0.0031776509618949324]
['accelerating', 3.0, 20.5, 0.9998627989042682, True, True, True, 0.0009236681330195007]
['stationary', 5.0, 30.5, 0.9810788905474227, True, True, True, 0.00036430461384305557]


## 6. Analytic Self-Check

This section verifies the exact constant-rate solution of the ODER law. It protects against validating an approximate shortcut instead of the current differential equation.

In [8]:
def fit_exact_constant_gamma(time, s_ret, gamma_grid, tau_char_grid):
    best = {"sse": float("inf"), "gamma": None, "tau_char": None}
    for gamma in gamma_grid:
        preds = np.array([exact_constant_gamma_solution(time, gamma, tc) for tc in tau_char_grid])
        sse = ((preds - s_ret) ** 2).sum(axis=1)
        idx = int(np.argmin(sse))
        if float(sse[idx]) < best["sse"]:
            best = {"sse": float(sse[idx]), "gamma": float(gamma), "tau_char": float(tau_char_grid[idx])}
    pred = exact_constant_gamma_solution(time, best["gamma"], best["tau_char"])
    best["r_squared"] = 1.0 - float(((s_ret - pred) ** 2).sum() / ((s_ret - s_ret.mean()) ** 2).sum())
    return best

gamma_true = 0.1
tau_char_true = 12.0
exact_trace = exact_constant_gamma_solution(TIME, gamma_true, tau_char_true)
exact_fit = fit_exact_constant_gamma(
    TIME,
    exact_trace,
    gamma_grid=np.linspace(0.08, 0.12, 401),
    tau_char_grid=np.linspace(10.0, 14.0, 401),
)
MANIFEST["pass_flags"]["analytic_constant_gamma_recovery"] = (
    abs(exact_fit["gamma"] - gamma_true) < 1e-12
    and abs(exact_fit["tau_char"] - tau_char_true) < 1e-12
)
print(exact_fit)
print("PASS" if MANIFEST["pass_flags"]["analytic_constant_gamma_recovery"] else "WARN")

{'sse': 0.0, 'gamma': 0.1, 'tau_char': 12.0, 'r_squared': 1.0}
PASS


## 7. Bootstrap Confidence Bands

The bootstrap perturbs retrieval rates under the frozen preset. It tests whether bounded retrieval and class separation survive the stated spectral-noise envelope.

In [9]:
bootstrap = {}
for name, cfg in OBSERVER_PRESET.items():
    base_gamma = cfg["gamma_fn"](TIME)
    traces = np.zeros((N_BOOTSTRAP, len(TIME)))
    for i in range(N_BOOTSTRAP):
        noisy_gamma = np.clip(base_gamma + rng.normal(0.0, BOOTSTRAP_GAMMA_NOISE, len(TIME)), 0.0, None)
        traces[i] = integrate_retrieval(TIME, noisy_gamma, cfg["tau_char"])
    bootstrap[name] = {
        "mean": traces.mean(axis=0),
        "lower": np.percentile(traces, 2.5, axis=0),
        "upper": np.percentile(traces, 97.5, axis=0),
        "bounded": bool((traces >= -1e-12).all() and (traces <= S_MAX + 1e-12).all()),
    }

MANIFEST["pass_flags"]["bootstrap_bounded"] = all(v["bounded"] for v in bootstrap.values())
for name in PREDICTED_ORDER:
    b = bootstrap[name]
    print(name, "final 95% CI", float(b["lower"][-1]), float(b["upper"][-1]), "bounded", b["bounded"])

free_falling final 95% CI 0.9999999874297658 0.9999999890286455 bounded True
accelerating final 95% CI 0.9998543889275074 0.9998717453240342 bounded True
stationary final 95% CI 0.9798699247986352 0.9819917022409959 bounded True


## 8. D=4/D=8 Finite-Resolution Robustness

This section verifies the finite-resolution proxy claim used in the paper. The check asks whether D=4 and D=8 preserve boundedness, monotonicity, and observer ordering.

`D` changes the resolution/noise envelope. It does not define a new entropy law and does not perform explicit tensor contractions.

In [10]:
def resolution_adjusted_trace(observer, bond_dimension):
    cfg = OBSERVER_PRESET[observer]
    base_gamma = cfg["gamma_fn"](TIME)
    resolution_factor = np.log(bond_dimension) / np.log(4.0)
    tau_char_eff = cfg["tau_char"] / np.sqrt(resolution_factor)
    noise_sigma = BOOTSTRAP_GAMMA_NOISE * (4.0 / bond_dimension)
    local_rng = np.random.default_rng(RANDOM_SEED + bond_dimension * 100 + len(observer))
    gamma_eff = np.clip(base_gamma + local_rng.normal(0.0, noise_sigma, len(TIME)), 0.0, None)
    s_ret = integrate_retrieval(TIME, gamma_eff, tau_char_eff)
    return s_ret, tau_char_eff

resolution = {}
resolution_rows = []
for observer in OBSERVER_PRESET:
    resolution[observer] = {}
    for d in BOND_DIMENSIONS:
        s_ret, tau_char_eff = resolution_adjusted_trace(observer, d)
        metrics = trace_metrics(TIME, s_ret, tau_char_eff)
        resolution[observer][d] = {"s_ret": s_ret, "tau_char_eff": tau_char_eff, "metrics": metrics}
        resolution_rows.append([observer, d, tau_char_eff, metrics["tau_rh"], metrics["final_entropy"], metrics["bounded"], metrics["monotone"]])
        print(resolution_rows[-1])

resolution_csv = save_csv(
    TABLE_DIR / "d4_d8_resolution_robustness.csv",
    ["observer", "D", "tau_char_eff", "tau_rh", "final_entropy", "bounded", "monotone"],
    resolution_rows,
)
MANIFEST["artifacts"]["d4_d8_resolution_robustness"] = artifact_path(resolution_csv)

d4_d8_pass = True
for d in BOND_DIMENSIONS:
    d_metrics = {observer: resolution[observer][d]["metrics"] for observer in OBSERVER_PRESET}
    d4_d8_pass = d4_d8_pass and order_pass_from_metrics(d_metrics)
    d4_d8_pass = d4_d8_pass and all(m["bounded"] and m["monotone"] for m in d_metrics.values())

MANIFEST["pass_flags"]["d4_d8_resolution_robustness"] = bool(d4_d8_pass)
print("PASS" if d4_d8_pass else "WARN")

['stationary', 4, np.float64(5.0), 30.700000000000003, 0.9802073410337314, True, True]
['stationary', 8, np.float64(4.08248290463863), 30.1, 0.9815288678867587, True, True]
['free_falling', 4, np.float64(2.0), 12.9, 0.9999999875218122, True, True]
['free_falling', 8, np.float64(1.6329931618554523), 12.8, 0.9999999885789029, True, True]
['accelerating', 4, np.float64(3.0), 20.900000000000002, 0.999854918906107, True, True]
['accelerating', 8, np.float64(2.4494897427831783), 20.200000000000003, 0.9998702376539612, True, True]
PASS


## 9. Adversarial Nulls

The null suite asks whether common non-ODER alternatives can fake the current signature set.

Each control breaks one part of the interpretation while preserving enough surface structure to make the comparison fair.

### 9.1 Matched Saturating Envelope Null

This control breaks the ODER generating law while preserving saturation. Logistic, Gompertz, stretched-exponential, and Hill-type traces are matched to the same ceiling and approximate retrieval horizons.

The output asks whether generic saturation can reproduce ordering, inverse-map stability, and low residual structure.

In [11]:
def logistic_trace(time, tau_mid, width):
    y = 1.0 / (1.0 + np.exp(-(time - tau_mid) / width))
    y0 = y[0]
    return np.clip((y - y0) / (1.0 - y0), 0.0, 1.0)

def gompertz_trace(time, tau_mid, width):
    b = math.log(1.0 / TAU_RH_THRESHOLD)
    y = np.exp(-b * np.exp(-(time - tau_mid) / width))
    y0 = y[0]
    return np.clip((y - y0) / (1.0 - y0), 0.0, 1.0)

def stretched_exp_trace(time, tau_mid, beta):
    scale = tau_mid / ((-math.log(1.0 - TAU_RH_THRESHOLD)) ** (1.0 / beta))
    return np.clip(1.0 - np.exp(-((time / scale) ** beta)), 0.0, 1.0)

def hill_trace(time, tau_mid, power):
    k = tau_mid * ((1.0 / TAU_RH_THRESHOLD - 1.0) ** (1.0 / power))
    return np.clip((time ** power) / (time ** power + k ** power), 0.0, 1.0)

SATURATION_FAMILIES = {
    "logistic": lambda center: logistic_trace(TIME, center, 3.0),
    "gompertz": lambda center: gompertz_trace(TIME, center, 5.0),
    "stretched_exp": lambda center: stretched_exp_trace(TIME, center, 1.35),
    "hill": lambda center: hill_trace(TIME, center, 3.0),
}

def calibrate_family_to_retrieval_horizon(family_fn, target_tau_rh):
    centers = np.linspace(max(0.1, target_tau_rh * 0.2), min(TIME_MAX, target_tau_rh * 1.8 + 5.0), 500)
    best = None
    for center in centers:
        trace = family_fn(center)
        horizon = tau_rh(TIME, trace)
        if horizon is None:
            continue
        err = abs(horizon - target_tau_rh)
        if best is None or err < best[0]:
            best = (err, float(center), trace, float(horizon))
    if best is None:
        trace = family_fn(target_tau_rh)
        horizon = tau_rh(TIME, trace)
        return trace, float(target_tau_rh), horizon
    return best[2], best[1], best[3]

matched_null_rows = []
matched_null = {}
for family, fn in SATURATION_FAMILIES.items():
    matched_null[family] = {}
    for observer in PREDICTED_ORDER:
        target = core[observer]["metrics"]["tau_rh"]
        s_ret, calibrated_center, calibrated_horizon = calibrate_family_to_retrieval_horizon(fn, target)
        # Same small noise envelope, then monotone repair to preserve saturation fairness.
        noisy = np.clip(s_ret + rng.normal(0.0, 0.002, len(TIME)), 0.0, S_MAX)
        noisy = np.maximum.accumulate(noisy)
        metrics = trace_metrics(TIME, noisy, OBSERVER_PRESET[observer]["tau_char"])
        metrics["inverse_stable"] = inverse_stable(metrics)
        matched_null[family][observer] = {"s_ret": noisy, "metrics": metrics}
        matched_null_rows.append([
            family,
            observer,
            target,
            calibrated_center,
            calibrated_horizon,
            metrics["tau_rh"],
            metrics["inverse_stable"],
            metrics["inverse_gamma_cv"],
            metrics["inverse_roughness"],
        ])

matched_csv = save_csv(
    TABLE_DIR / "matched_saturating_envelope_null.csv",
    ["family", "observer", "target_tau_rh", "calibrated_center", "calibrated_tau_rh", "observed_tau_rh_after_noise", "inverse_stable", "inverse_gamma_cv", "inverse_roughness"],
    matched_null_rows,
)
MANIFEST["artifacts"]["matched_saturating_envelope_null"] = artifact_path(matched_csv)

families_passing_full_signature = 0
for family in SATURATION_FAMILIES:
    fam_metrics = {observer: matched_null[family][observer]["metrics"] for observer in PREDICTED_ORDER}
    full = order_pass_from_metrics(fam_metrics) and all(m["inverse_stable"] for m in fam_metrics.values())
    families_passing_full_signature += int(full)

MANIFEST["pass_flags"]["matched_saturating_null_rejected"] = families_passing_full_signature <= MATCHED_NULL_ALLOWED_FULL_SIGNATURES
MANIFEST["nulls_executed"].append("Matched Saturating Envelope Null")
print("families passing full ODER signature:", families_passing_full_signature)
print("PASS" if MANIFEST["pass_flags"]["matched_saturating_null_rejected"] else "WARN")

families passing full ODER signature: 0
PASS


### 9.2 Shared-Observer Envelope Null

This control breaks observer-specific retrieval geometry. All observer classes receive the same underlying envelope; only amplitude and noise vary.

It tests whether the reported class separation could be visual scaling rather than observer-indexed dynamics.

In [12]:
shared_base = exact_constant_gamma_solution(TIME, gamma=0.115, tau_char=4.0)
shared = {}
shared_rows = []
for observer, amp, noise in [
    ("free_falling", 1.00, 0.002),
    ("accelerating", 0.995, 0.003),
    ("stationary", 0.990, 0.004),
]:
    local_rng = np.random.default_rng(RANDOM_SEED + len(observer))
    s_ret = np.clip(amp * shared_base + local_rng.normal(0.0, noise, len(TIME)), 0.0, S_MAX)
    s_ret = np.maximum.accumulate(s_ret)
    metrics = trace_metrics(TIME, s_ret, OBSERVER_PRESET[observer]["tau_char"])
    metrics["inverse_stable"] = inverse_stable(metrics)
    shared[observer] = {"s_ret": s_ret, "metrics": metrics}
    shared_rows.append([observer, metrics["tau_rh"], metrics["inverse_stable"], metrics["inverse_gamma_cv"]])
    print(shared_rows[-1])

shared_csv = save_csv(
    TABLE_DIR / "shared_observer_envelope_null.csv",
    ["observer", "tau_rh", "inverse_stable", "inverse_gamma_cv"],
    shared_rows,
)
MANIFEST["artifacts"]["shared_observer_envelope_null"] = artifact_path(shared_csv)
shared_order = order_pass_from_metrics({k: v["metrics"] for k, v in shared.items()})
shared_separation = len({shared[k]["metrics"]["tau_rh"] for k in shared}) == len(shared)
MANIFEST["pass_flags"]["shared_observer_envelope_null_rejected"] = not (shared_order and shared_separation)
MANIFEST["nulls_executed"].append("Shared-Observer Envelope Null")
print("shared_order", shared_order, "shared_separation", shared_separation)
print("PASS" if MANIFEST["pass_flags"]["shared_observer_envelope_null_rejected"] else "WARN")

['free_falling', 22.900000000000002, True, 1.2227760508387122]
['accelerating', 23.1, False, 1.6411924371081825]
['stationary', 23.1, False, 2.096824455553253]
shared_order False shared_separation False
PASS


### 9.3 Observer-Label Permutation Null

This control breaks the mapping between trajectory and observer label. It preserves the trajectories and noise but randomly assigns the labels.

The output is a calibration baseline: the probability that strict expected ordering appears by chance.

In [13]:
base_tau = {name: core[name]["metrics"]["tau_rh"] for name in PREDICTED_ORDER}
success = 0
labels = list(PREDICTED_ORDER)
for _ in range(N_PERMUTATIONS):
    perm = rng.permutation(labels)
    assigned = {label: base_tau[source] for label, source in zip(labels, perm)}
    vals = [assigned[name] for name in PREDICTED_ORDER]
    success += int(all(a < b for a, b in zip(vals, vals[1:])))

permutation_p = (success + 1) / (N_PERMUTATIONS + 1)
chance_baseline = 1.0 / math.factorial(len(PREDICTED_ORDER))
permutation_rows = [["expected_order_count", success], ["n_permutations", N_PERMUTATIONS], ["p_value", permutation_p], ["chance_baseline", chance_baseline]]
permutation_csv = save_csv(TABLE_DIR / "observer_label_permutation_null.csv", ["metric", "value"], permutation_rows)
MANIFEST["artifacts"]["observer_label_permutation_null"] = artifact_path(permutation_csv)
MANIFEST["pass_flags"]["observer_label_permutation_quantified"] = abs(permutation_p - chance_baseline) < LABEL_PERMUTATION_BASELINE_TOLERANCE
MANIFEST["nulls_executed"].append("Observer-Label Permutation Null")
print("ordering count", success, "of", N_PERMUTATIONS, "p", permutation_p, "chance baseline", chance_baseline)
print("PASS" if MANIFEST["pass_flags"]["observer_label_permutation_quantified"] else "WARN")

ordering count 346 of 2000 p 0.17341329335332334 chance baseline 0.16666666666666666
PASS


### 9.4 Proper-Time Jitter Null

This control perturbs proper-time alignment while preserving monotonicity and saturation. It tests whether retrieval-horizon ordering is fragile to timing uncertainty.

In [14]:
jitter_success = 0
jitter_rows = []
for i in range(N_JITTER):
    jitter_metrics = {}
    for observer in PREDICTED_ORDER:
        cfg = OBSERVER_PRESET[observer]
        tau_char_j = max(0.2, cfg["tau_char"] + rng.normal(0.0, JITTER_SIGMA))
        time_j = np.maximum.accumulate(TIME + rng.normal(0.0, JITTER_SIGMA, len(TIME)))
        time_j[0] = 0.0
        time_j = time_j + np.arange(len(time_j)) * 1e-6
        gamma = cfg["gamma_fn"](TIME)
        s = integrate_retrieval(time_j, gamma, tau_char_j)
        jitter_metrics[observer] = trace_metrics(time_j, s, tau_char_j)
    ok = order_pass_from_metrics(jitter_metrics)
    jitter_success += int(ok)
    if i < 10:
        jitter_rows.append([i] + [jitter_metrics[o]["tau_rh"] for o in PREDICTED_ORDER] + [ok])

jitter_rate = jitter_success / N_JITTER
jitter_csv = save_csv(
    TABLE_DIR / "proper_time_jitter_null.csv",
    ["sample", "free_falling_tau_rh", "accelerating_tau_rh", "stationary_tau_rh", "order_preserved"],
    jitter_rows,
)
MANIFEST["artifacts"]["proper_time_jitter_null"] = artifact_path(jitter_csv)
MANIFEST["pass_flags"]["proper_time_jitter_robust"] = jitter_rate >= ORDERING_PRESERVATION_THRESHOLD
MANIFEST["nulls_executed"].append("Proper-Time Jitter Null")
print("ordering survival rate", jitter_rate)
print("PASS" if MANIFEST["pass_flags"]["proper_time_jitter_robust"] else "WARN")

ordering survival rate 1.0
PASS


### 9.5 Non-Gap Dynamics Null

This control breaks the ODER gap structure. It generates monotone saturating trajectories from dynamics not of the form `dS = gamma(tau)(Smax-S)f(tau)` and then applies the ODER inverse map anyway.

The expected outcome is degraded inverse-map diagnostics.

In [15]:
def integrate_non_gap(time, drive):
    s = np.zeros_like(time)
    for i in range(1, len(time)):
        dt = time[i] - time[i - 1]
        s[i] = s[i - 1] + drive[i - 1] * dt
    if s.max() > 0:
        s = s / s.max()
    return np.clip(s, 0.0, S_MAX)

non_gap = {}
non_gap_rows = []
for observer in PREDICTED_ORDER:
    if observer == "free_falling":
        drive = 0.02 + 0.20 / (1 + np.exp(-(TIME - 8) / 1.2))
    elif observer == "accelerating":
        drive = 0.05 + 0.002 * TIME + 0.01 * np.sin(TIME / 3)
    else:
        drive = 0.035 + 0.015 * np.exp(-TIME / 12)
    s = integrate_non_gap(TIME, np.clip(drive, 0, None))
    metrics = trace_metrics(TIME, s, OBSERVER_PRESET[observer]["tau_char"])
    metrics["inverse_stable"] = inverse_stable(metrics)
    non_gap[observer] = {"s_ret": s, "metrics": metrics}
    non_gap_rows.append([observer, metrics["tau_rh"], metrics["inverse_stable"], metrics["inverse_gamma_cv"], metrics["inverse_roughness"]])
    print(non_gap_rows[-1])

non_gap_csv = save_csv(
    TABLE_DIR / "non_gap_dynamics_null.csv",
    ["observer", "tau_rh", "inverse_stable", "inverse_gamma_cv", "inverse_roughness"],
    non_gap_rows,
)
MANIFEST["artifacts"]["non_gap_dynamics_null"] = artifact_path(non_gap_csv)
non_gap_stable_count = sum(int(non_gap[o]["metrics"]["inverse_stable"]) for o in PREDICTED_ORDER)
MANIFEST["pass_flags"]["non_gap_dynamics_null_rejected"] = non_gap_stable_count <= NON_GAP_MAX_STABLE_INVERSES
MANIFEST["nulls_executed"].append("Non-Gap Dynamics Null")
print("non-gap stable inverse count", non_gap_stable_count)
print("PASS" if MANIFEST["pass_flags"]["non_gap_dynamics_null_rejected"] else "WARN")

['free_falling', 45.800000000000004, False, 3.31233902362809, 0.00025125631820271943]
['accelerating', 46.5, False, 4.010305513020563, 0.0001909293899029288]
['stationary', 44.6, False, 2.4715482832175883, 0.00024198861729322202]
non-gap stable inverse count 0
PASS


## 10. Figures

Each figure is generated after the tables and controls. Captions state what is shown, why it matters, and which null or diagnostic it is meant to stress.

### Figure Captions

**Retrieval rate profiles.** Shows the frozen observer-class `gamma(tau)` profiles. This matters because observer separation is defined at the retrieval-rate level before entropy curves are plotted. Shared-envelope or label-permutation nulls should not preserve the same trajectory-to-observer mapping.

**Entropy retrieval bootstrap bands.** Shows mean retrieval curves and 95% bootstrap bands. This matters because boundedness and 90% Retrieval Horizon ordering should survive the declared spectral-noise envelope. Generic saturation may match a single curve but should fail the full signature set.

**Inverse retrieval profiles.** Shows `gamma(tau)` recovered from generated `S_ret(tau)`. This matters because the ODER inverse map should recover the known generating profile with low error. Non-Gap Dynamics Null traces should degrade this diagnostic.

**D=4/D=8 finite-resolution robustness.** Shows whether the finite-resolution proxy settings used in the paper preserve the retrieval structure. This matters because the paper cites D=4/D=8 robustness. The check should fail if ordering or boundedness depends on a single resolution setting.

**Matched saturating envelope nulls.** Shows generic saturating alternatives against the ODER accelerating trace. This matters because visual saturation alone is not the ODER signature. These nulls should fail the full signature set.

**Schematic ODER g2 envelope.** Shows the suppressive tanh-modulated two-time envelope used as a schematic correlation signature. This matters as a bridge from retrieval dynamics to measurable envelope structure. It is not an empirical data plot.

In [16]:
plt.figure(figsize=(9, 5))
for name, cfg in OBSERVER_PRESET.items():
    plt.plot(TIME, core[name]["gamma"], label=cfg["label"])
plt.xlabel("Proper time tau")
plt.ylabel("Retrieval rate gamma(tau)")
plt.title("Observer-Class Retrieval Rate Profiles")
plt.grid(alpha=0.25)
plt.legend()
gamma_fig = FIGURE_DIR / "retrieval_rate_profiles.png"
plt.savefig(gamma_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["retrieval_rate_profiles"] = artifact_path(gamma_fig)

plt.figure(figsize=(9, 5))
for name, cfg in OBSERVER_PRESET.items():
    b = bootstrap[name]
    plt.plot(TIME, b["mean"], label=cfg["label"])
    plt.fill_between(TIME, b["lower"], b["upper"], alpha=0.18)
plt.axhline(TAU_RH_THRESHOLD, color="black", linestyle="--", linewidth=1, label="90% Retrieval Horizon threshold")
plt.xlabel("Proper time tau")
plt.ylabel("Retrieved entropy S_ret / S_max")
plt.title("Entropy Retrieval Curves with Bootstrap Bands")
plt.grid(alpha=0.25)
plt.legend()
entropy_fig = FIGURE_DIR / "entropy_retrieval_bootstrap_bands.png"
plt.savefig(entropy_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["entropy_retrieval_bootstrap_bands"] = artifact_path(entropy_fig)

plt.figure(figsize=(9, 5))
for name, cfg in OBSERVER_PRESET.items():
    plt.plot(TIME, core[name]["gamma_inv"], label=f"{cfg['label']} inverted")
plt.xlabel("Proper time tau")
plt.ylabel("Inverted gamma(tau)")
plt.title("Inverse Retrieval Map")
plt.grid(alpha=0.25)
plt.legend()
inverse_fig = FIGURE_DIR / "inverse_gamma_profiles.png"
plt.savefig(inverse_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["inverse_gamma_profiles"] = artifact_path(inverse_fig)

plt.figure(figsize=(9, 5))
for name, cfg in OBSERVER_PRESET.items():
    for d, linestyle in zip(BOND_DIMENSIONS, ["-", "--"]):
        plt.plot(TIME, resolution[name][d]["s_ret"], linestyle=linestyle, label=f"{cfg['label']} D={d}")
plt.axhline(TAU_RH_THRESHOLD, color="black", linestyle=":", linewidth=1)
plt.xlabel("Proper time tau")
plt.ylabel("Retrieved entropy S_ret / S_max")
plt.title("D=4/D=8 Finite-Resolution Robustness")
plt.grid(alpha=0.25)
plt.legend(ncol=2, fontsize=8)
resolution_fig = FIGURE_DIR / "d4_d8_resolution_robustness.png"
plt.savefig(resolution_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["d4_d8_resolution_robustness_figure"] = artifact_path(resolution_fig)

plt.figure(figsize=(9, 5))
for family in SATURATION_FAMILIES:
    plt.plot(TIME, matched_null[family]["accelerating"]["s_ret"], label=family)
plt.plot(TIME, core["accelerating"]["s_ret"], color="black", linewidth=2, label="ODER accelerating")
plt.xlabel("Proper time tau")
plt.ylabel("Retrieved entropy S_ret / S_max")
plt.title("Matched Saturating Envelope Nulls")
plt.grid(alpha=0.25)
plt.legend()
matched_fig = FIGURE_DIR / "matched_saturating_envelope_nulls.png"
plt.savefig(matched_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["matched_saturating_envelope_nulls_figure"] = artifact_path(matched_fig)

t1, t2 = np.meshgrid(np.linspace(0, 30, 160), np.linspace(0, 30, 160), indexing="ij")
tau_char_g2 = OBSERVER_PRESET["accelerating"]["tau_char"]
g2 = (1.0 - np.tanh(t1 / tau_char_g2)) * (1.0 - np.tanh(t2 / tau_char_g2))
plt.figure(figsize=(7, 6))
plt.imshow(g2, origin="lower", extent=[0, 30, 0, 30], aspect="auto")
plt.colorbar(label="g2 envelope amplitude")
plt.xlabel("t1")
plt.ylabel("t2")
plt.title("Schematic ODER g2 Envelope")
g2_fig = FIGURE_DIR / "g2_envelope_heatmap.png"
plt.savefig(g2_fig, dpi=200, bbox_inches="tight")
plt.show()
MANIFEST["artifacts"]["g2_envelope_heatmap"] = artifact_path(g2_fig)

## 11. Verification Report

In [17]:
claim_map = [
    ("Core ODER traces remain bounded and monotone.", "core_observer_summary", MANIFEST["pass_flags"]["core_bounded"] and MANIFEST["pass_flags"]["core_monotone"]),
    ("Observer retrieval horizons follow free_falling < accelerating < stationary.", "core_observer_summary", MANIFEST["pass_flags"]["core_order"]),
    ("The ODER inverse map recovers gamma(tau) with low error on generated retrieval traces.", "inverse_gamma_profiles", MANIFEST["pass_flags"]["core_inverse_recovery"]),
    ("Bootstrap confidence bands use 200 traces per observer in publication mode.", "entropy_retrieval_bootstrap_bands", N_BOOTSTRAP == 200),
    ("D=4/D=8 finite-resolution proxy settings preserve the retrieval structure.", "d4_d8_resolution_robustness", MANIFEST["pass_flags"]["d4_d8_resolution_robustness"]),
    ("Matched generic saturation does not reproduce the full ODER signature set.", "matched_saturating_envelope_null", MANIFEST["pass_flags"]["matched_saturating_null_rejected"]),
    ("A Shared-Observer Envelope Null does not reproduce observer-class separation.", "shared_observer_envelope_null", MANIFEST["pass_flags"]["shared_observer_envelope_null_rejected"]),
    ("Observer-Label Permutation Null reports the chance ordering baseline.", "observer_label_permutation_null", MANIFEST["pass_flags"]["observer_label_permutation_quantified"]),
    ("Proper-time jitter preserves retrieval-horizon ordering.", "proper_time_jitter_null", MANIFEST["pass_flags"]["proper_time_jitter_robust"]),
    ("A Non-Gap Dynamics Null degrades ODER inverse-map diagnostics.", "non_gap_dynamics_null", MANIFEST["pass_flags"]["non_gap_dynamics_null_rejected"]),
]

report = {
    "notebook_artifact": NOTEBOOK_ARTIFACT,
    "notebook_version": NOTEBOOK_VERSION,
    "analysis_preset": ANALYSIS_PRESET,
    "run_mode": RUN_MODE,
    "random_seed": RANDOM_SEED,
    "created_utc": MANIFEST["created_utc"],
    "python": MANIFEST["python"],
    "numpy": MANIFEST["numpy"],
    "n_bootstrap": N_BOOTSTRAP,
    "n_permutations": N_PERMUTATIONS,
    "n_jitter": N_JITTER,
    "observer_summary": {
        name: {
            "tau_char": OBSERVER_PRESET[name]["tau_char"],
            "tau_rh": core[name]["metrics"]["tau_rh"],
            "final_entropy": core[name]["metrics"]["final_entropy"],
            "inverse_gamma_median_abs_error": core[name]["metrics"]["inverse_gamma_median_abs_error"],
        }
        for name in PREDICTED_ORDER
    },
    "exact_constant_gamma_fit": exact_fit,
    "permutation_p_value": permutation_p,
    "jitter_order_survival_rate": jitter_rate,
    "thresholds": MANIFEST["thresholds"],
    "nulls_executed": MANIFEST["nulls_executed"],
    "pass_flags": MANIFEST["pass_flags"],
    "claim_map": [
        {"claim": claim, "artifact": artifact, "status": "PASS" if ok else "WARN"}
        for claim, artifact, ok in claim_map
    ],
    "artifacts": MANIFEST["artifacts"],
    "boundary_notes": [
        "Synthetic/proxy verification only.",
        "No external astrophysical data are ingested.",
        "D=4/D=8 is treated as a finite-resolution proxy check.",
        "Adversarial nulls test generic saturation and structure-breaking alternatives without adding new physical models.",
    ],
}

manifest_path = OUTPUT_DIR / "validation_manifest.json"
report_path = OUTPUT_DIR / "verification_report.md"
MANIFEST["artifacts"]["validation_manifest"] = artifact_path(manifest_path)
MANIFEST["artifacts"]["verification_report"] = artifact_path(report_path)
report["artifacts"] = MANIFEST["artifacts"]
manifest_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

lines = ["# ODER-BH Verification Report", ""]
lines.append(f"- artifact: `{NOTEBOOK_ARTIFACT}`")
lines.append(f"- version: `{NOTEBOOK_VERSION}`")
lines.append(f"- analysis preset: `{ANALYSIS_PRESET}`")
lines.append(f"- run mode: `{RUN_MODE}`")
lines.append(f"- bootstrap traces per observer: `{N_BOOTSTRAP}`")
lines.append(f"- observer-label permutations: `{N_PERMUTATIONS}`")
lines.append("")
lines.append("## Thresholds")
for key, val in MANIFEST["thresholds"].items():
    lines.append(f"- {key}: `{val}`")
lines.append("")
lines.append("## Observer Summary")
for name in PREDICTED_ORDER:
    vals = report["observer_summary"][name]
    lines.append(f"- {name}: 90% Retrieval Horizon={vals['tau_rh']}, final S={vals['final_entropy']:.4f}, inverse gamma MAE={vals['inverse_gamma_median_abs_error']:.5g}")
lines.append("")
lines.append("## Adversarial Summary")
lines.append(f"- matched saturation families passing full signature: `{families_passing_full_signature}`")
lines.append(f"- Shared-Observer Envelope Null order reproduced: `{shared_order}`")
lines.append(f"- observer-label permutation p-value: `{permutation_p:.4f}`")
lines.append(f"- proper-time jitter ordering survival: `{jitter_rate:.3f}`")
lines.append(f"- Non-Gap Dynamics Null stable inverse count: `{non_gap_stable_count}`")
lines.append(f"- nulls executed: `{', '.join(MANIFEST['nulls_executed'])}`")
lines.append("")
lines.append("## Pass Flags")
for key, val in MANIFEST["pass_flags"].items():
    lines.append(f"- {key}: {'PASS' if val else 'WARN'}")
lines.append("")
lines.append("## Claim Map")
for item in report["claim_map"]:
    lines.append(f"- {item['status']}: {item['claim']} -> {item['artifact']}")
lines.append("")
lines.append("## Boundary Notes")
for note in report["boundary_notes"]:
    lines.append(f"- {note}")
lines.append("- Individual nulls may pass isolated diagnostics, but no adversarial family reproduces the full joint ODER signature across observer classes.")

report_path.write_text("\n".join(lines), encoding="utf-8")

print("\n".join(lines))
print(f"\nWrote {artifact_path(manifest_path)}")
print(f"Wrote {artifact_path(report_path)}")

# ODER-BH Verification Report

- artifact: `ODER_BH_verification_artifact_v2`
- version: `2.0`
- analysis preset: `ODER_BH_RETRIEVAL_V2`
- run mode: `publication`
- bootstrap traces per observer: `200`
- observer-label permutations: `2000`

## Thresholds
- bootstrap_traces_publication: `200`
- retrieval_horizon_order: `['free_falling', 'accelerating', 'stationary']`
- inverse_recovery_median_abs_error_lt: `0.005`
- ordering_preservation_ge: `0.95`
- matched_null_allowed_full_signatures: `0`
- non_gap_max_stable_inverses: `1`
- label_permutation_baseline_tolerance: `0.05`
- inverse_finite_fraction_min: `0.55`
- inverse_negative_fraction_max: `0.02`
- inverse_gamma_cv_max: `1.25`
- inverse_roughness_max: `0.035`
- bond_dimensions: `[4, 8]`

## Observer Summary
- free_falling: 90% Retrieval Horizon=12.9, final S=1.0000, inverse gamma MAE=0.0031777
- accelerating: 90% Retrieval Horizon=20.5, final S=0.9999, inverse gamma MAE=0.00092367
- stationary: 90% Retrieval Horizon=30.5, final S=0.98

## 12. Extension Notes

To extend this notebook:

- declare a new analysis preset before changing scientific parameters;
- add a new observer class by registering `gamma_fn`, `tau_char`, and label in `OBSERVER_PRESET`;
- add a new adversarial null by stating what it breaks and what it preserves;
- keep generated figures in Section 10 and claim mappings in Section 11;
- keep boundary notes explicit when moving from synthetic/proxy verification to external data.

Design principle: a reader should be able to stop at the final report, know the result, and walk backward to the exact preset, control, code, and artifact that produced it.